# Logistic Regression với PyTorch
## Ví dụ với dataset Iris

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Load và chuẩn bị dữ liệu

In [ ]:
# Load dữ liệu
data = pd.read_csv('../data/IRIS.csv')
print("Thông tin dữ liệu:")
print(data.head())
print(f"\nShape: {data.shape}")
print(f"\nCác loài: {data['species'].unique()}")

In [ ]:
# Tách features và labels
X = data.iloc[:, :-1].values  # Lấy 4 cột đầu (features)
y = data.iloc[:, -1].values   # Lấy cột cuối (species)

# Encode labels thành số
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Classes: {label_encoder.classes_}")
print(f"Encoded labels: {np.unique(y)}")

In [ ]:
# Chuẩn hóa features (quan trọng cho neural networks)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape}, {y_train.shape}")
print(f"Test set: {X_test.shape}, {y_test.shape}")

In [ ]:
# Chuyển sang PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

print(f"Train tensors: {X_train_tensor.shape}, {y_train_tensor.shape}")
print(f"Test tensors: {X_test_tensor.shape}, {y_test_tensor.shape}")

## 2. Định nghĩa Logistic Regression Model

Logistic Regression = Linear Layer + Activation
- Input: 4 features
- Output: 3 classes (Iris-setosa, Iris-versicolor, Iris-virginica)

In [ ]:
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(LogisticRegressionModel, self).__init__()
        # Chỉ có 1 layer linear duy nhất
        self.linear = nn.Linear(input_dim, output_dim)
        # Softmax được tính trong loss function (CrossEntropyLoss)
    
    def forward(self, x):
        # Output: logits (chưa qua softmax)
        return self.linear(x)

# Khởi tạo model
input_size = X_train.shape[1]  # 4 features
num_classes = len(np.unique(y))  # 3 classes

model = LogisticRegressionModel(input_size, num_classes)
print(model)
print(f"\nTổng số parameters: {sum(p.numel() for p in model.parameters())}")

## 3. Định nghĩa Loss Function và Optimizer

In [ ]:
# Loss function cho multi-class classification
# CrossEntropyLoss = LogSoftmax + NLLLoss
criterion = nn.CrossEntropyLoss()

# Optimizer - Stochastic Gradient Descent
learning_rate = 0.01
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

# Hoặc có thể dùng Adam optimizer (thường tốt hơn)
# optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"Loss function: {criterion}")
print(f"Optimizer: {optimizer}")

## 4. Training Loop

In [ ]:
# Hyperparameters
num_epochs = 1000

# Lưu loss để vẽ biểu đồ
losses = []
accuracies = []

# Training loop
for epoch in range(num_epochs):
    # 1. Forward pass
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    
    # 2. Backward pass
    optimizer.zero_grad()  # Reset gradients
    loss.backward()         # Tính gradients
    optimizer.step()        # Update weights
    
    # 3. Tính accuracy
    with torch.no_grad():
        # Dự đoán class (lấy index của giá trị lớn nhất)
        _, predicted = torch.max(outputs.data, 1)
        correct = (predicted == y_train_tensor).sum().item()
        accuracy = correct / y_train_tensor.size(0) * 100
    
    # Lưu metrics
    losses.append(loss.item())
    accuracies.append(accuracy)
    
    # In kết quả mỗi 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.2f}%')

print("\n✅ Training hoàn thành!")

## 5. Visualize Training Process

In [ ]:
# Vẽ biểu đồ Loss và Accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(losses, label='Training Loss', color='red', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Loss vs Epochs', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curve
ax2.plot(accuracies, label='Training Accuracy', color='blue', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Accuracy vs Epochs', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Đánh giá Model trên Test Set

In [ ]:
# Chuyển model sang evaluation mode
model.eval()

with torch.no_grad():
    # Dự đoán trên test set
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs.data, 1)
    
    # Tính accuracy
    correct = (predicted == y_test_tensor).sum().item()
    total = y_test_tensor.size(0)
    test_accuracy = correct / total * 100
    
    print(f"Test Accuracy: {test_accuracy:.2f}%")
    print(f"Correct predictions: {correct}/{total}")
    print(f"\nDự đoán: {predicted.numpy()}")
    print(f"Thực tế:  {y_test_tensor.numpy()}")

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

cm = confusion_matrix(y_test_tensor.numpy(), predicted.numpy())

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test_tensor.numpy(), predicted.numpy(), 
                          target_names=label_encoder.classes_))

## 7. Prediction trên dữ liệu mới

In [ ]:
# Tạo một mẫu mới để dự đoán
new_sample = np.array([[5.1, 3.5, 1.4, 0.2]])  # Iris-setosa

# Chuẩn hóa (dùng scaler đã training)
new_sample_scaled = scaler.transform(new_sample)

# Chuyển sang tensor
new_sample_tensor = torch.FloatTensor(new_sample_scaled)

# Dự đoán
model.eval()
with torch.no_grad():
    output = model(new_sample_tensor)
    
    # Softmax để lấy xác suất
    probabilities = torch.softmax(output, dim=1)
    
    # Lấy class có xác suất cao nhất
    _, predicted_class = torch.max(output, 1)
    
    print(f"Mẫu dữ liệu: {new_sample[0]}")
    print(f"\nXác suất dự đoán:")
    for i, species in enumerate(label_encoder.classes_):
        print(f"  {species}: {probabilities[0][i].item():.4f} ({probabilities[0][i].item()*100:.2f}%)")
    
    print(f"\n🎯 Dự đoán: {label_encoder.classes_[predicted_class.item()]}")

## 8. Xem Weights và Bias của Model

In [ ]:
# Xem parameters của model
print("Model Parameters:")
print("="*50)
for name, param in model.named_parameters():
    print(f"\n{name}:")
    print(f"Shape: {param.shape}")
    print(f"Values:\n{param.data}")

## 9. Save và Load Model

In [ ]:
# Save model
torch.save(model.state_dict(), '../models/logistic_regression_iris.pth')
print("✅ Model đã được lưu tại: '../models/logistic_regression_iris.pth'")

# Load model
# loaded_model = LogisticRegressionModel(input_size, num_classes)
# loaded_model.load_state_dict(torch.load('../models/logistic_regression_iris.pth'))
# loaded_model.eval()
# print("✅ Model đã được load thành công!")

---
## 📚 Tóm tắt: Các bước Logistic Regression với PyTorch

1. **Chuẩn bị dữ liệu**: Load, encode labels, normalize, split train/test
2. **Chuyển sang Tensor**: Convert NumPy array sang PyTorch tensor
3. **Định nghĩa Model**: Tạo class kế thừa `nn.Module`
4. **Loss & Optimizer**: CrossEntropyLoss + SGD/Adam
5. **Training Loop**: Forward → Loss → Backward → Update weights
6. **Evaluation**: Test accuracy, confusion matrix
7. **Prediction**: Dự đoán trên dữ liệu mới
8. **Save/Load**: Lưu và load model

### Key Concepts:
- **Logistic Regression** = Linear layer duy nhất
- **CrossEntropyLoss** = LogSoftmax + NLLLoss (tự động tính softmax)
- **Softmax** chuyển logits thành probabilities
- **torch.max()** lấy class có probability cao nhất
